In [8]:
# TASK 2: BASELINE eLCS EXECUTION ON RAW DATASET

import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from skeLCS import eLCS

# Load Raw Phase 1 Dataset
df = pd.read_csv('ardd_fatalities.csv', low_memory=False)

# Drop structural artifact column
if 'Unnamed: 23' in df.columns:
    df = df.drop(columns=['Unnamed: 23'])

# Filter Target Population (Youth: 17-25 vs Older: 26+)
df = df[~df['Age'].isin([-9, '-9'])].copy()
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df = df.dropna(subset=['Age'])

df = df[df['Age'] >= 17].copy()
df['Target_Youth'] = (df['Age'] <= 25).astype(int)

# Select baseline feature subset
feature_cols = [
    'State', 'Speed Limit', 'Crash Type', 'Bus Involvement', 
    'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement', 
    'Dayweek', 'Time of day'
]
X_df = df[feature_cols].copy()
y = df['Target_Youth'].values

# Minimal Processing for LCS Compatibility (Basic Numeric Encoding)
for col in X_df.columns:
    X_df[col] = X_df[col].replace([-9, '-9', '-9.0'], np.nan)
    X_df[col] = X_df[col].astype(str).astype('category').cat.codes

X = X_df.values

# Train / Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Execute Baseline eLCS Model
elcs_params = {
    'learning_iterations': 5000,
    'N': 500,
    'p_spec': 0.5,
    'fitness_mode': 'accuracy'
}

print("Running Baseline eLCS...")
# Updated eLCS parameters compatible with skeLCS
elcs_params = {
    'learning_iterations': 5000,
    'N': 500,
    'p_spec': 0.5
}

print("Running Baseline eLCS...")
model = eLCS(**elcs_params)
start_time = time.time()
model.fit(X_train, y_train)
print(f"Training Time: {time.time() - start_time:.2f} seconds")

# Evaluation
y_pred = model.predict(X_test)
print("\n=== BASELINE eLCS RESULTS ===\n")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Running Baseline eLCS...
Running Baseline eLCS...
Training Time: 1.94 seconds

=== BASELINE eLCS RESULTS ===

Accuracy:  0.7230
Precision: 0.7149
Recall:    0.7230
F1-Score:  0.6125

Confusion Matrix:
 [[7339   19]
 [2809   43]]


In [2]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-elcs"])

  Using cached scikit_elcs-1.2.4-py3-none-any.whl


0